In [0]:

dbutils.widgets.text("environment", "dev")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
spark.sql(f"drop table if exists {env}_banking.metadata.tables")
spark.sql(f"drop table if exists {env}_banking.metadata.table_parameters")
spark.sql(f"drop table if exists {env}_banking.metadata.table_watermarks")
spark.sql(f"drop table if exists {env}_banking.metadata.pipeline_runs")
spark.sql(f"drop table if exists {env}_banking.metadata.batch_control")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.batch_control (
    ingestion_run_id STRING,
    batch_id STRING,
    source_table STRING,
    target_table STRING,
    target_layer STRING,
    load_type STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    record_count BIGINT,
    error_message STRING
) USING DELTA        
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.tables (
    table_id            INT,
    table_name          STRING,
    source_system       STRING,        -- sqlserver / blob
    source_schema       STRING,        -- dbo (null for blob)
    source_table        STRING,        -- table name (null for blob)
    source_path         STRING,        -- blob path (null for sqlserver)
    target_layer        STRING,        -- silver/gold
    bronze_schema       STRING,        -- bronze
    silver_schema       STRING,        -- silver
    gold_schema         STRING,        -- gold
    active_flag         BOOLEAN,
    load_order          INT,
    created_at          TIMESTAMP
)
USING DELTA;
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.table_parameters (
    table_id            INT,
    parameter_name      STRING,        -- load_type / primary_key / watermark_column
    parameter_value     STRING,
    created_at          TIMESTAMP
)
USING DELTA;
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env}_banking.metadata.table_watermarks (
    process_name            STRING,
    table_name              STRING,
    last_source_timestamp   TIMESTAMP,     -- flexible type storage
    last_ingestion_run_id   STRING,
    last_updated_at         TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
    INSERT INTO {env}_banking.metadata.table_watermarks VALUES
    ('bronze', 'customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'transaction', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'account', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'account_customer_history', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'transaction_line', '1900-01-01 00:00:00', NULL, current_timestamp()),
    ('bronze', 'daily_account_balance', '1900-01-01 00:00:00', NULL, current_timestamp())
""")

In [0]:
spark.sql(f"""
   INSERT INTO {env}_banking.metadata.tables VALUES
    (1, 'branch', 'sqlserver', 'data_source', 'branch', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),
    (2, 'customer_history', 'sqlserver', 'data_source', 'customer_history', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),
    (3, 'transaction', 'sqlserver', 'data_source', 'transaction', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),
    (4, 'account', 'sqlserver', 'data_source', 'account', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),
    (5, 'product', 'sqlserver', 'data_source', 'product', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 5, current_timestamp()),
    (6, 'account_customer_history', 'sqlserver', 'data_source', 'account_customer_history', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 6, current_timestamp()),
    (7, 'transaction_line', 'sqlserver', 'data_source', 'transaction_line', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 7, current_timestamp()),
    (8, 'daily_account_balance', 'sqlserver', 'daily_account_balance', 'branch', NULL, 'bronze', 'bronze', 'silver', NULL, TRUE, 8, current_timestamp())
""")

In [0]:
%sql
USE IDENTIFIER(:environment || '_banking.metadata')

In [0]:
%sql
INSERT INTO table_parameters VALUES

-- ================= CUSTOMERS HISTORY =================
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'customer_id,effective_from_date', current_timestamp()),
(2, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'transaction_id', current_timestamp()),
(3, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= ACCOUNTS =================
(4, 'load_type', 'MERGE', current_timestamp()),
(4, 'primary_key', 'account_id', current_timestamp()),
(4, 'watermark_column', 'source_modified_at', current_timestamp()),

-- ================= BRANCH =================
(1, 'load_type', 'FULL', current_timestamp()),
(1, 'primary_key', 'branch_id', current_timestamp()),

-- ================= PRODUCT =================
(5, 'load_type', 'FULL', current_timestamp()),
(5, 'primary_key', 'product_id', current_timestamp()),

-- ================= ACCOUNT CUSTOMERS HISTORY =================
(6, 'load_type', 'MERGE', current_timestamp()),
(6, 'primary_key', 'account_id,customer_id,effective_from_date', current_timestamp()),
(6, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= TRANSACTIONS LINE=================
(7, 'load_type', 'APPEND', current_timestamp()),
(7, 'primary_key', 'transaction_id,line_number', current_timestamp()),
(7, 'watermark_column', 'source_modified_at', current_timestamp()),

--  ================= DAILY ACCOUNT BALANCE =================
(8, 'load_type', 'MERGE', current_timestamp()),
(8, 'primary_key', 'balance_date,account_id', current_timestamp()),
(8, 'watermark_column', 'source_modified_at', current_timestamp());